In [22]:
from sentence_transformers import SentenceTransformer
import sqlite3
import sqlite_vec
import numpy as np
from ingest import load_faq_data
from rag_helper import RAGHelper
import faiss
import os
from dotenv import load_dotenv
from anthropic import Anthropic
from ingest import load_faq_data, build_faiss_index

In [17]:
load_dotenv()

True

In [20]:
anthropic_client = Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))

In [3]:
model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [11]:
documents = load_faq_data()

In [5]:
db = sqlite3.connect("vector_search.db")
db.enable_load_extension(True)
sqlite_vec.load(db)
db.enable_load_extension(False)

In [7]:
q1_embedding = model.encode("Can I run Kafka?")
query_vector = q1_embedding.astype(np.float32).tobytes()

In [ ]:
rows = db.execute(
    """
    SELECT rowid, distance
    FROM faq_vectors
    WHERE embedding MATCH ? AND course = ? AND k = 5
    ORDER BY distance
""",
    (query_vector, "llm-zoomcamp"),
).fetchall()

In [12]:
top_5_docs = [documents[rowid - 1] for rowid, _ in rows]

In [13]:
top_5_docs

[{'id': '193612db63',
  'course': 'llm-zoomcamp',
  'section': 'Module 3: Orchestration',
  'question': "Why do we need orchestration / Kestra — can't I just run the code in a notebook?",
  'answer': "Notebooks are great for learning and experimenting, but real AI workflows need more than a script that runs once: scheduling, retries, monitoring, secret management, and reliably chaining tasks together. That's what an orchestrator like Kestra provides.\n\nIn this module Kestra is also the vehicle for the AI techniques the course is teaching: AI Copilot to generate flows from natural language, RAG to ground responses in real data, and AI agents that decide which tools to call. The goal is to see how AI fits into production-style workflows, not just notebook cells.\n\nKestra's AI plugins also work with any major provider (OpenAI, Gemini, Anthropic, and more), so you can swap providers in a flow without changing anything else. See the [module intro](https://github.com/DataTalksClub/llm-zoom

In [15]:
class RAGVectorSearch(RAGHelper):
    def __init__(self, embedder, documents, **kwargs):
        super().__init__(**kwargs)
        self.embedder = embedder
        self.documents = documents

    def search(self, query, num_results=5):
        query_vector = self.embedder.encode(query).astype(np.float32).reshape(1, -1)
        course_ids = np.array(
            [i for i, doc in enumerate(self.documents) if doc["course"] == self.course],
            dtype=np.int32,
        )
        id_selector = faiss.IDSelectorArray(course_ids)
        params = faiss.SearchParameters(sel=id_selector)

        _, indices = self.index.search(query_vector, num_results, params=params)
        return [self.documents[i] for i in indices[0] if i != -1]

In [23]:
index_faiss = build_faiss_index(model=model, documents=documents)

In [24]:
vector_assistant = RAGVectorSearch(
    embedder=model, index=index_faiss, documents=documents, llm_client=anthropic_client
)

In [26]:
print(vector_assistant.rag("The program has already begun. Can I still sign up?"))

Yes, you can still sign up! Even though the program has already begun, you're welcome to join. However, if you want to receive a certificate, you'll need to submit your project **while submissions are still being accepted**.

You can start learning and submitting homework right away without waiting for a confirmation email. Registration is not checked against any official list—it's simply used to gauge interest.
